# Signal and Image Processing — Milestone 3 Final Code

Final submitted model: **Custom CNN V4**.

This notebook uses a manually configured CNN made from standard Keras layers such as convolution, pooling, batch normalization, dropout, squeeze-and-excitation attention, and dense layers. The dataset files are not modified; images are only resized in memory while being loaded for training and testing.


In [ ]:
# ============================================================
# Cell 1 — Set zip paths
# ============================================================
# Upload the train and test zip files directly to Colab.
# Make sure the file names match exactly.

TRAIN_ZIP = '/content/train-20260516T125715Z-3-001.zip'
TEST_ZIP  = '/content/test-20260516T125715Z-3-001.zip'


In [ ]:
# ============================================================
# Cell 2 — Imports, setup, extraction, and dataset audit
# ============================================================
import os
import zipfile
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, regularizers
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay
)

print('TensorFlow version:', tf.__version__)

SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DATA_DIR = '/content/ball_dataset'
os.makedirs(DATA_DIR, exist_ok=True)

def unzip_if_needed(zip_path, extract_to):
    assert os.path.exists(zip_path), f'Zip file not found: {zip_path}'
    with zipfile.ZipFile(zip_path, 'r') as z:
        top_folder = z.namelist()[0].split('/')[0]
        target_folder = os.path.join(extract_to, top_folder)

        if os.path.exists(target_folder) and len(os.listdir(target_folder)) > 0:
            print(f'{top_folder}/ already extracted. Skipping.')
        else:
            print(f'Extracting {zip_path} ...')
            t0 = time.time()
            z.extractall(extract_to)
            print(f'Done in {time.time() - t0:.1f} s')

unzip_if_needed(TRAIN_ZIP, DATA_DIR)
unzip_if_needed(TEST_ZIP, DATA_DIR)

TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR  = os.path.join(DATA_DIR, 'test')

assert os.path.isdir(TRAIN_DIR), f'Train folder not found: {TRAIN_DIR}'
assert os.path.isdir(TEST_DIR), f'Test folder not found: {TEST_DIR}'

IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

def count_images(root):
    counts = {}
    for cls in sorted(os.listdir(root)):
        cls_dir = os.path.join(root, cls)
        if os.path.isdir(cls_dir):
            counts[cls] = len([
                f for f in os.listdir(cls_dir)
                if f.lower().endswith(IMG_EXTS)
            ])
    return counts

train_counts = count_images(TRAIN_DIR)
test_counts = count_images(TEST_DIR)
classes = sorted(train_counts.keys())

print('\nDataset distribution:')
print(f'{"Class":<20}{"Train":>8}{"Test":>8}{"Total":>8}')
print('-' * 44)
for cls in classes:
    tr = train_counts.get(cls, 0)
    te = test_counts.get(cls, 0)
    print(f'{cls:<20}{tr:>8}{te:>8}{tr+te:>8}')
print('-' * 44)
print(f'{"TOTAL":<20}{sum(train_counts.values()):>8}{sum(test_counts.values()):>8}{sum(train_counts.values())+sum(test_counts.values()):>8}')


In [ ]:
# ============================================================
# Cell 3 — Load high-resolution train, validation, and test datasets
# ============================================================
# The original image files are not changed.
# image_size only resizes temporary tensors in memory before passing them to the CNN.

IMG_SIZE = (256, 256)
BATCH_SIZE = 24
NUM_CLASSES = 6
AUTOTUNE = tf.data.AUTOTUNE

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.15,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.15,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    shuffle=False,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_ds_raw.class_names
test_file_paths = test_ds_raw.file_paths

print('\nClass order used by the model:')
for i, name in enumerate(class_names):
    print(i, name)

train_ds = train_ds_raw.cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds = val_ds_raw.cache().prefetch(AUTOTUNE)
test_ds = test_ds_raw.cache().prefetch(AUTOTUNE)


In [ ]:
# ============================================================
# Cell 4 — Show sample training images
# ============================================================
plt.figure(figsize=(12, 8))

for images, labels in train_ds_raw.take(1):
    for i in range(min(12, len(images))):
        ax = plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[int(np.argmax(labels[i]))])
        plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Cell 5 — Define Custom CNN V4
# ============================================================

def conv_bn_act(x, filters, kernel_size=3, strides=1, weight_decay=8e-5, name=None):
    x = layers.Conv2D(
        filters,
        kernel_size,
        strides=strides,
        padding='same',
        use_bias=False,
        kernel_regularizer=regularizers.l2(weight_decay),
        name=None if name is None else name + '_conv'
    )(x)

    x = layers.BatchNormalization(
        name=None if name is None else name + '_bn'
    )(x)

    x = layers.Activation(
        'swish',
        name=None if name is None else name + '_swish'
    )(x)

    return x


def squeeze_excitation_block(x, ratio=8, name='se'):
    channels = int(x.shape[-1])

    s = layers.GlobalAveragePooling2D(name=name + '_gap')(x)
    s = layers.Dense(
        max(channels // ratio, 8),
        activation='relu',
        name=name + '_dense1'
    )(s)
    s = layers.Dense(
        channels,
        activation='sigmoid',
        name=name + '_dense2'
    )(s)
    s = layers.Reshape((1, 1, channels), name=name + '_reshape')(s)

    return layers.Multiply(name=name + '_scale')([x, s])


def residual_se_block(x, filters, strides=1, dropout=0.0, weight_decay=8e-5, name='block'):
    shortcut = x

    x = conv_bn_act(
        x,
        filters,
        kernel_size=3,
        strides=strides,
        weight_decay=weight_decay,
        name=name + '_conv1'
    )

    x = layers.Conv2D(
        filters,
        3,
        padding='same',
        use_bias=False,
        kernel_regularizer=regularizers.l2(weight_decay),
        name=name + '_conv2'
    )(x)
    x = layers.BatchNormalization(name=name + '_bn2')(x)

    x = squeeze_excitation_block(x, ratio=8, name=name + '_se')

    if int(shortcut.shape[-1]) != filters or strides != 1:
        shortcut = layers.Conv2D(
            filters,
            1,
            strides=strides,
            padding='same',
            use_bias=False,
            kernel_regularizer=regularizers.l2(weight_decay),
            name=name + '_shortcut_conv'
        )(shortcut)
        shortcut = layers.BatchNormalization(name=name + '_shortcut_bn')(shortcut)

    x = layers.Add(name=name + '_add')([x, shortcut])
    x = layers.Activation('swish', name=name + '_out')(x)

    if dropout > 0:
        x = layers.Dropout(dropout, name=name + '_dropout')(x)

    return x


def build_custom_cnn_v4(input_shape=(256, 256, 3), num_classes=6):
    inputs = tf.keras.Input(shape=input_shape, name='input_image')

    # Augmentation is applied during training only and does not alter stored files.
    x = layers.RandomFlip('horizontal', name='aug_flip')(inputs)
    x = layers.RandomRotation(0.05, name='aug_rotation')(x)
    x = layers.RandomZoom(0.08, name='aug_zoom')(x)
    x = layers.RandomContrast(0.12, name='aug_contrast')(x)

    x = layers.Rescaling(1./255, name='rescale')(x)

    # Stem
    x = conv_bn_act(x, 32, 3, 1, name='stem1')
    x = conv_bn_act(x, 32, 3, 1, name='stem2')
    x = layers.MaxPooling2D(pool_size=2, name='stem_pool')(x)

    # Main feature extraction stages
    x = residual_se_block(x, 64,  strides=1, dropout=0.05, name='s1_b1')
    x = residual_se_block(x, 64,  strides=1, dropout=0.05, name='s1_b2')

    x = residual_se_block(x, 128, strides=2, dropout=0.08, name='s2_b1')
    x = residual_se_block(x, 128, strides=1, dropout=0.08, name='s2_b2')

    x = residual_se_block(x, 192, strides=2, dropout=0.12, name='s3_b1')
    x = residual_se_block(x, 192, strides=1, dropout=0.12, name='s3_b2')

    x = residual_se_block(x, 256, strides=2, dropout=0.18, name='s4_b1')
    x = residual_se_block(x, 256, strides=1, dropout=0.18, name='s4_b2')

    x = residual_se_block(x, 384, strides=2, dropout=0.22, name='s5_b1')

    # Classification head
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.BatchNormalization(name='head_bn')(x)
    x = layers.Dropout(0.45, name='head_drop1')(x)

    x = layers.Dense(
        512,
        activation='swish',
        kernel_regularizer=regularizers.l2(8e-5),
        name='dense_512'
    )(x)
    x = layers.Dropout(0.35, name='head_drop2')(x)

    x = layers.Dense(
        256,
        activation='swish',
        kernel_regularizer=regularizers.l2(8e-5),
        name='dense_256'
    )(x)
    x = layers.Dropout(0.25, name='head_drop3')(x)

    outputs = layers.Dense(
        num_classes,
        activation='softmax',
        name='class_probabilities'
    )(x)

    return tf.keras.Model(inputs, outputs, name='Custom_CNN_V4_SE_HighRes_From_Scratch')


model = build_custom_cnn_v4(IMG_SIZE + (3,), NUM_CLASSES)

loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.02)

try:
    optimizer = tf.keras.optimizers.AdamW(learning_rate=5e-4, weight_decay=8e-5)
except Exception:
    optimizer = tf.keras.optimizers.Adam(learning_rate=5e-4)

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=['accuracy']
)

model.summary()
print('\nTotal parameters:', model.count_params())


In [ ]:
# ============================================================
# Cell 6 — Train Custom CNN V4
# ============================================================

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        '/content/best_custom_cnn_v4.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=16,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

start_time = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=callbacks
)

training_time = time.time() - start_time

print(f'Custom CNN V4 training time: {training_time:.2f} seconds ({training_time/60:.2f} minutes)')

model = tf.keras.models.load_model('/content/best_custom_cnn_v4.keras')


In [ ]:
# ============================================================
# Cell 7 — Plot training curves
# ============================================================

plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='train accuracy')
plt.plot(history.history['val_accuracy'], label='validation accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Custom CNN V4 Training and Validation Accuracy')
plt.legend()
plt.grid(True)
plt.savefig('/content/custom_cnn_v4_accuracy_curve.png', dpi=150, bbox_inches='tight')
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Custom CNN V4 Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.savefig('/content/custom_cnn_v4_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================
# Cell 8 — Final evaluation on unchanged test set
# ============================================================

y_true_onehot = np.concatenate([labels.numpy() for images, labels in test_ds_raw], axis=0)
y_true = np.argmax(y_true_onehot, axis=1)

pred_probs = model.predict(test_ds_raw, verbose=1)
y_pred = np.argmax(pred_probs, axis=1)

test_accuracy = accuracy_score(y_true, y_pred)
macro_precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
macro_recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

print('=' * 80)
print(f'Custom CNN V4 accuracy : {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')
print(f'Macro precision        : {macro_precision:.4f}')
print(f'Macro recall           : {macro_recall:.4f}')
print(f'Macro F1-score         : {macro_f1:.4f}')
print('=' * 80)

print('\nClassification report:')
print(classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0))


In [ ]:
# ============================================================
# Cell 9 — Confusion matrix
# ============================================================

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(9, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues', values_format='d', xticks_rotation=45, colorbar=False)
plt.title('Custom CNN V4 — Confusion Matrix')
plt.tight_layout()
plt.savefig('/content/custom_cnn_v4_confusion_matrix_counts.png', dpi=150, bbox_inches='tight')
plt.show()

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(9, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=class_names)
disp.plot(cmap='Blues', values_format='.2f', xticks_rotation=45, colorbar=False)
plt.title('Custom CNN V4 — Row-Normalized Confusion Matrix')
plt.tight_layout()
plt.savefig('/content/custom_cnn_v4_confusion_matrix_normalized.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================
# Cell 10 — Top 3 probabilities for selected test images
# ============================================================

from PIL import Image

indices = np.linspace(0, len(test_file_paths)-1, 12, dtype=int)

plt.figure(figsize=(16, 12))

for plot_i, idx in enumerate(indices):
    img_path = test_file_paths[idx]
    true_label = class_names[y_true[idx]]
    pred_label = class_names[y_pred[idx]]

    top3 = np.argsort(pred_probs[idx])[-3:][::-1]
    top3_text = '\n'.join([
        f'{class_names[j]}: {pred_probs[idx][j]*100:.1f}%'
        for j in top3
    ])

    img = Image.open(img_path).convert('RGB').resize(IMG_SIZE)

    ax = plt.subplot(3, 4, plot_i + 1)
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'True: {true_label}\nPred: {pred_label}\n{top3_text}', fontsize=9)

plt.tight_layout()
plt.savefig('/content/custom_cnn_v4_top3_probabilities.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 3 probabilities for selected test images:')
print('-' * 70)

for idx in indices:
    file_name = os.path.basename(test_file_paths[idx])
    true_label = class_names[y_true[idx]]

    top3 = np.argsort(pred_probs[idx])[-3:][::-1]

    print(f'Image: {file_name} | True class: {true_label}')
    for rank, j in enumerate(top3, start=1):
        print(f'  {rank}. {class_names[j]:<20} {pred_probs[idx][j]*100:6.2f}%')
    print('-' * 70)


In [ ]:
# ============================================================
# Cell 11 — Save final summary
# ============================================================

summary_text = f"""
Milestone 3 Custom CNN V4 Summary
=================================

Model name: {model.name}
Input size: {IMG_SIZE[0]} x {IMG_SIZE[1]} x 3
Number of classes: {NUM_CLASSES}
Class order: {class_names}

Total parameters: {model.count_params()}

Test accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)
Macro precision: {macro_precision:.4f}
Macro recall: {macro_recall:.4f}
Macro F1-score: {macro_f1:.4f}

Training runtime: {training_time:.2f} seconds ({training_time/60:.2f} minutes)

Saved figures:
- /content/custom_cnn_v4_accuracy_curve.png
- /content/custom_cnn_v4_loss_curve.png
- /content/custom_cnn_v4_confusion_matrix_counts.png
- /content/custom_cnn_v4_confusion_matrix_normalized.png
- /content/custom_cnn_v4_top3_probabilities.png
"""

print(summary_text)

with open('/content/custom_cnn_v4_summary.txt', 'w') as f:
    f.write(summary_text)

print('Saved /content/custom_cnn_v4_summary.txt')
